# 01 Data Collection & Feature Engineering
This notebook handles the extraction of Spatio-Temporal data from Google Earth Engine (GEE) and local feature engineering for Karachi monitoring stations.

In [ ]:
import ee
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import holidays

try:
    ee.Initialize(project='gen-lang-client-0478151371')
    print("GEE Initialized Successfully")
except Exception as e:
    print(f"GEE Initialization Failed: {e}")

## 1. Define Stations & Parameters

In [ ]:
stations = {
    'Site_Area_Ind': [66.99, 24.90],
    'Korangi_Ind': [67.12, 24.83],
    'Port_Qasim_Ind': [67.34, 24.78],
    'Nazimabad_Res': [67.03, 24.91],
    'Gulshan_Res': [67.09, 24.92],
    'Malir_Res': [67.19, 24.89],
    'Saddar_Comm': [67.02, 24.86],
    'Clifton_Coast': [67.03, 24.81]
}

start_date = '2019-01-01'
end_date = '2024-01-01'
karachi_roi = ee.Geometry.Polygon([[66.7, 24.7], [67.5, 24.7], [67.5, 25.1], [66.7, 25.1]])

## 2. Sentinel-5P Extraction (AER, NO2, SO2, CO)

In [ ]:
def extract_s5p(collection_id, band_name, suffix):
    col = ee.ImageCollection(collection_id) \
        .filterDate(start_date, end_date) \
        .filterBounds(karachi_roi) \
        .select(band_name)
    
    def map_stations(img):
        return img.reduceRegions(
            collection=ee.FeatureCollection([ee.Feature(ee.Geometry.Point(c), {'station': n}) for n, c in stations.items()]),
            reducer=ee.Reducer.mean(),
            scale=1113
        ).map(lambda f: f.set('date', img.date().format('YYYY-MM-dd')))

    flat_col = col.map(map_stations).flatten()
    return flat_col

# Queueing S5P Exports
s5p_aer = extract_s5p('COPERNICUS/S5P/NRTI/L3_AER_AI', 'aerosol_index_354_388', 'aer')
s5p_no2 = extract_s5p('COPERNICUS/S5P/NRTI/L3_NO2', 'NO2_column_number_density', 'no2')
# Add others as needed...

## 3. MODIS & ERA5 Engineering

In [ ]:
modis = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
    .filterDate(start_date, end_date) \
    .filterBounds(karachi_roi) \
    .select('Optical_Depth_055') \
    .map(lambda img: img.multiply(0.001).set('date', img.date().format('YYYY-MM-dd')))

era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
    .filterDate(start_date, end_date) \
    .filterBounds(karachi_roi)

def engineer_era5(img):
    u = img.select('u_component_of_wind_10m')
    v = img.select('v_component_of_wind_10m')
    ws = u.hypot(v).rename('wind_speed')
    wd = u.atan2(v).rename('wind_dir')
    t = img.select('temperature_2m')
    td = img.select('dewpoint_temperature_2m')
    rh = ee.Image(100).multiply(ee.Image(10).pow(ee.Image(17.625).multiply(td.subtract(273.15)).divide(td.subtract(273.15).add(243.04)))).divide(ee.Image(10).pow(ee.Image(17.625).multiply(t.subtract(273.15)).divide(t.subtract(273.15).add(243.04)))).rename('rh')
    return img.addBands([ws, wd, rh])

era5_engineered = era5.map(engineer_era5)

## 4. Export Tasks (Run these to queue on GEE)

In [ ]:
def export_to_drive(collection, name):
    task = ee.batch.Export.table.toDrive(
        collection=collection,
        description=name,
        fileFormat='CSV'
    )
    # task.start() # Uncomment to start
    print(f"Task {name} queued")

# export_to_drive(s5p_aer, 'karachi_s5p_aer')
# ... repeat for other sources

## 5. Offline Feature Engineering (Pakistan Specific)
Run this after downloading the CSVs.

In [ ]:
def add_pakistan_features(df):
    # Eid and Ramadan Logic (Estimated for 2019-2024)
    ramadan_dates = [
        ('2019-05-05', '2019-06-04'), ('2020-04-23', '2020-05-23'),
        ('2021-04-12', '2021-05-12'), ('2022-04-01', '2022-05-01'),
        ('2023-03-22', '2023-04-20'), ('2024-03-10', '2024-04-09')
    ]
    
    df['is_ramadan'] = 0
    for start, end in ramadan_dates:
        df.loc[(df['date'] >= start) & (df['date'] <= end), 'is_ramadan'] = 1
        
    # Cyclical Encoding
    df['date_dt'] = pd.to_datetime(df['date'])
    df['month_sin'] = np.sin(2 * np.pi * df['date_dt'].dt.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['date_dt'].dt.month / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['date_dt'].dt.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['date_dt'].dt.dayofweek / 7)
    
    return df

print("Feature engineering functions ready.")